In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import datetime, pandas

from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Union
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import TimeSeriesSplit
import warnings
from abc import ABC, abstractmethod

import kaggle_evaluation.default_inference_server

from catboost import CatBoostRegressor, Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoLars
from sklearn.linear_model import Lasso
from sklearn.linear_model import BayesianRidge
from sklearn.linear_model import TweedieRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestCentroid

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import CategoricalNB

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

In [2]:
train = pandas.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv').dropna()
test = pandas.read_csv('/kaggle/input/hull-tactical-market-prediction/test.csv').dropna()

In [3]:
train.columns

Index(['date_id', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'E1',
       'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19',
       'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'I1', 'I2', 'I3',
       'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'M1', 'M10', 'M11', 'M12', 'M13',
       'M14', 'M15', 'M16', 'M17', 'M18', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7',
       'M8', 'M9', 'P1', 'P10', 'P11', 'P12', 'P13', 'P2', 'P3', 'P4', 'P5',
       'P6', 'P7', 'P8', 'P9', 'S1', 'S10', 'S11', 'S12', 'S2', 'S3', 'S4',
       'S5', 'S6', 'S7', 'S8', 'S9', 'V1', 'V10', 'V11', 'V12', 'V13', 'V2',
       'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'forward_returns',
       'risk_free_rate', 'market_forward_excess_returns'],
      dtype='object')

In [4]:
scalers = {}

def preprocessing(data, typ):
    # Define all available features organized by category
    common_data_features = [
        # Dummy/Binary features
        'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9',
        # Economic features
        'E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'E10',
        'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20',
        # Interest Rate features
        'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9',
        # Market features
        'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'M10',
        'M11', 'M12', 'M13', 'M14', 'M15', 'M16', 'M17', 'M18',
        # Price features
        'P8', 'P9', 'P10', 'P12', 'P13',
        # Sentiment features
        'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12',
        # Volatility features
        'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13'
    ]
    
    # Select features and target
    if typ == "train":
        selected_columns = common_data_features + ["forward_returns"]
    else:
        selected_columns = common_data_features
    
    # Filter only available columns
    available_columns = [col for col in selected_columns if col in data.columns]
    data = data[available_columns].copy()
    
    # Feature categories for different treatment
    dummy_features = [f for f in data.columns if f.startswith('D')]
    continuous_features = [f for f in data.columns if not f.startswith('D') and f != 'forward_returns']
    
    # Handle missing values intelligently
    # For dummy features: fill with mode (most common value)
    for feature in dummy_features:
        if data[feature].isnull().any():
            mode_val = data[feature].mode()[0] if not data[feature].mode().empty else 0
            data[feature].fillna(mode_val, inplace=True)
    
    # For continuous features: use forward fill then median
    for feature in continuous_features:
        if data[feature].isnull().any():
            # Forward fill first (good for time series)
            data[feature].fillna(method='ffill', inplace=True)
            # Fill remaining with median
            data[feature].fillna(data[feature].median(), inplace=True)
    
    # Handle target variable if present
    if 'forward_returns' in data.columns and data['forward_returns'].isnull().any():
        data['forward_returns'].fillna(data['forward_returns'].median(), inplace=True)
    
    # Remove outliers using IQR method (cap instead of remove)
    for feature in continuous_features:
        Q1 = data[feature].quantile(0.25)
        Q3 = data[feature].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Cap outliers
        data[feature] = np.clip(data[feature], lower_bound, upper_bound)
    
    # Scale continuous features (not dummy features)
    if continuous_features:
        if typ == "train":
            # Fit scaler on training data
            scalers['continuous'] = RobustScaler()
            data[continuous_features] = scalers['continuous'].fit_transform(data[continuous_features])
        else:
            # Use fitted scaler on test data
            if 'continuous' in scalers:
                data[continuous_features] = scalers['continuous'].transform(data[continuous_features])
    
    # Create engineered features for BOTH train and test
    # Market-Volatility interaction
    if 'M1' in data.columns and 'V1' in data.columns:
        data['M1_V1_interaction'] = data['M1'] * data['V1']
        # Handle nulls in interaction feature
        if data['M1_V1_interaction'].isnull().any():
            data['M1_V1_interaction'].fillna(0, inplace=True)  # Interaction nulls = 0
    
    # Sentiment-Market interaction
    if 'S1' in data.columns and 'M1' in data.columns:
        data['S1_M1_interaction'] = data['S1'] * data['M1']
        # Handle nulls in interaction feature
        if data['S1_M1_interaction'].isnull().any():
            data['S1_M1_interaction'].fillna(0, inplace=True)  # Interaction nulls = 0
    
    # Add simple lag feature for key indicator
    if 'M1' in data.columns:
        data['M1_lag1'] = data['M1'].shift(1)
        # Handle nulls in lag feature differently for train vs test
        if typ == "train":
            # For training: use median of the lag column itself
            if data['M1_lag1'].isnull().any():
                data['M1_lag1'].fillna(data['M1_lag1'].median(), inplace=True)
        else:
            # For test: use the current M1 value for first row, then median for others
            if data['M1_lag1'].isnull().any():
                # Fill first row with current M1 value
                first_null_idx = data['M1_lag1'].isnull().idxmax()
                if pd.notna(data.loc[first_null_idx, 'M1']):
                    data.loc[first_null_idx, 'M1_lag1'] = data.loc[first_null_idx, 'M1']
                # Fill remaining with median
                data['M1_lag1'].fillna(data['M1_lag1'].median(), inplace=True)
    
    # Final comprehensive null check for ALL columns
    for column in data.columns:
        if data[column].isnull().any():
            if column.startswith('D'):  # Dummy features
                # For test set, use mode or fallback to 0
                mode_values = data[column].mode()
                fill_value = mode_values[0] if len(mode_values) > 0 else 0
                data[column].fillna(fill_value, inplace=True)
            
            elif column == 'forward_returns':  # Target variable (only in train)
                data[column].fillna(data[column].median(), inplace=True)
            
            elif '_interaction' in column:  # Engineered interaction features
                data[column].fillna(0, inplace=True)  # Missing interactions = 0
            
            elif '_lag' in column:  # Engineered lag features
                # Use forward fill first, then median
                data[column].fillna(method='ffill', inplace=True)
                data[column].fillna(data[column].median(), inplace=True)
            
            else:  # All other continuous features
                if typ == "test":
                    # For test set: forward fill first (time series), then median
                    data[column].fillna(method='ffill', inplace=True)
                    data[column].fillna(data[column].median(), inplace=True)
                else:
                    # For train set: use median directly
                    data[column].fillna(data[column].median(), inplace=True)
    
    return data
    
train = preprocessing(train, "train")

train_split, val_split = train_test_split(
    train, test_size=0.01, random_state=42
)

X_train = train_split.drop(columns=["forward_returns"])
X_test = val_split.drop(columns=["forward_returns"])
y_train = train_split['forward_returns']
y_test = val_split['forward_returns']

In [5]:
X_train

,D1,D2,D3,D4,D5,D6,D7,D8,D9,E1,...,V7,V8,V9,V10,V11,V12,V13,M1_V1_interaction,S1_M1_interaction,M1_lag1
7206,0,0,0,0,0,0,0,0,0,-0.639437,...,-0.782864,1.056414,-0.652250,-0.689856,0.320134,0.065678,-0.664302,-0.722862,-0.858975,-0.752708
7034,0,0,0,1,1,0,0,1,0,-1.158324,...,-0.894496,1.089322,-0.905749,-0.716788,1.074497,0.890537,-0.686074,-1.340825,-0.910768,-1.149593
7093,0,0,0,1,0,0,0,1,0,-1.230533,...,0.198884,1.013432,-0.021112,-0.213271,0.365772,0.781780,-1.913443,-0.329604,-0.076790,-0.423649
7588,0,0,0,1,0,0,0,0,0,1.070281,...,-0.143651,-0.081934,-0.274307,-0.392097,0.017450,-0.485169,0.072475,-0.033085,0.319171,-0.428005
7552,0,0,0,1,0,-1,0,0,0,0.699723,...,-0.462211,0.004701,-0.354454,-0.317464,0.039597,-0.691384,-0.129933,0.008156,0.011854,-0.085749
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8099,0,0,0,1,0,-1,0,0,0,-0.269405,...,1.241195,0.497649,1.001933,0.914948,0.627517,-0.325565,1.974023,0.959046,0.444293,1.019398
8263,0,0,0,1,0,0,0,0,0,-0.224526,...,1.023961,0.113499,0.848902,0.840866,-0.145638,0.281780,1.361576,0.133211,-0.063729,1.397913
7829,0,0,0,1,1,0,0,1,0,-0.267145,...,1.200932,0.603089,1.177337,1.186553,0.688591,0.947034,2.086557,0.198335,0.152447,0.387606
8428,0,0,0,0,0,0,0,0,0,-0.115825,...,-0.517440,-0.669577,-0.506721,-0.384811,-0.763758,-0.223164,-0.443943,0.550853,0.408501,-0.629607


In [6]:
improved_catboost_params = {'iterations': 3000,
                            'learning_rate': 0.01,
                            'depth': 6,
                            'l2_leaf_reg': 5.0,
                            'min_child_samples': 100,
                            'colsample_bylevel': 0.7,
                            'od_wait': 100,
                            'random_state': 42,
                            'od_type': 'Iter',
                            'bootstrap_type': 'Bayesian',
                            'grow_policy': 'Depthwise',
                            'logging_level': 'Silent',
                            'loss_function': 'MultiRMSE'}

R_Forest_parm = {'n_estimators': 100,
                 'min_samples_split': 5,
                 'max_depth': 15,
                 'min_samples_leaf': 3,
                 'max_features': 'sqrt',
                 'random_state': 42}
        
Extra_parm = {'n_estimators': 100,
              'min_samples_split': 5,
              'max_depth': 12,
              'min_samples_leaf': 3,
              'max_features': 'sqrt',
              'random_state': 42}
        
XGB_R_parm = {"n_estimators": 1500,
              "learning_rate": 0.05, 
              "max_depth": 6,
              "subsample": 0.8, 
              "colsample_bytree": 0.7,
              "reg_alpha": 1.0,
              "reg_lambda": 1.0,
              "random_state": 42}

LGBM_R_parm = {"n_estimators": 1500,
               "learning_rate": 0.05,
               "num_leaves": 50,
               "max_depth": 8,
               "reg_alpha": 1.0,
               "reg_lambda": 1.0,
               "random_state": 42,
               'verbosity': -1}

DecisionTree = {'criterion': 'poisson',
                'max_depth': 6}

GB_parm = {"learning_rate": 0.1,
           "min_samples_split": 500,
           "min_samples_leaf": 50,
           "max_depth": 8,
           "max_features": 'sqrt',
           "subsample": 0.8,
           "random_state": 10}

CatBoost = CatBoostRegressor(**improved_catboost_params)
XGBoost = XGBRegressor(**XGB_R_parm)
LGBM = LGBMRegressor(**LGBM_R_parm)
RandomForest = RandomForestRegressor(**R_Forest_parm)
ExtraTrees = ExtraTreesRegressor(**Extra_parm)
GBRegressor = GradientBoostingRegressor(**GB_parm)
        
DTRegressor = DecisionTreeRegressor(**DecisionTree)


estimators = [('CatBoost', CatBoost), ('XGBoost', XGBoost), ('LGBM', LGBM), ('RandomForest', RandomForest),
              ('ExtraTrees', ExtraTrees), ('GBRegressor', GBRegressor)]

model = StackingRegressor(estimators, 
                          final_estimator = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0]), 
                          cv=3)
model.fit(X_train, y_train)

StackingRegressor(cv=3,
                  estimators=[('CatBoost',
                               <catboost.core.CatBoostRegressor object at 0x79e30897a650>),
                              ('XGBoost',
                               XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=0.7, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None, gamma=None,
                                            g...
                                                     min_samples_split=5,
                                                     random_state=42)),
                              ('ExtraTrees',
                               ExtraTreesRegressor(max_depth=12,
                                                   max_features='sqrt',
                                                   min_samples_leaf=3,
                                                   min_samples_split=5,
                                                   random_state=42)),
                              ('GBRegressor',
                               GradientBoostingRegressor(max_depth=8,
                                                         max_features='sqrt',
                                                         min_samples_leaf=50,
                                                         min_samples_split=500,
                                                         random_state=10,
                                                         subsample=0.8))],
                  final_estimator=RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0]))

In [7]:
def predict(test: pl.DataFrame) -> float:
    test = test.to_pandas()
    test = preprocessing(test, "test")
    raw_pred = model.predict(test)[0]
    return raw_pred

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))